In [1]:
import pandas as pd
import os

In [2]:
pd.set_option('display.max_columns', None)

### Files

See **parameter explanation for detail information

In [3]:
file_path = "../Data/parameters of interest.xlsx"
files = pd.ExcelFile(file_path).sheet_names

In [4]:
files

['enteredit_findings',
 'pathology',
 'pathology_findings',
 'patient_data_ie',
 'hormonal_mens',
 'risk_factors',
 'vitals',
 'patient_demo',
 'family_hx']

In [5]:
shared_path = "../Data/R3Data"

### 6. risk_factors
<span style="color: blue;">Change idx value **based on EHR file**</span>

In [6]:
idx = 6
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

risk_factors
['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'RISK_FACTOR_CD', 'RISK_FACTOR_NAME', 'RISK_SEQUENCE', 'EXAM_COMPLETED_DATE']


In [7]:
file_names = [
    str(file) + ".txt", 
    str(file) + ".csv",
    str(file) + ".csv"
]
file_names

['risk_factors.txt', 'risk_factors.csv', 'risk_factors.csv']

### Cancer

In [8]:
study = "Cancer"
folder = ["R3_3787_Lee_Cancer_Extract_Files", "R3_3787_Lee_Data_Cancer_20240509", "R3_3787_Lee_Data_Cancer_20250912"]

In [9]:
final_unique_dfs = [] # List to hold the de-duplicated data from each file

# 2. Process each file
for i, file_name in enumerate(file_names):
    file_extension = os.path.splitext(file_name)[1].lower()
    file_path = os.path.join(shared_path, study, folder[i], file_name)
    
    if file_extension == '.csv':
        try:
            current_df = pd.read_csv(file_path, encoding='latin-1')
        except:
            print('file not exist in:', folder[i])
    elif file_extension == '.txt':
        try:
            current_df = pd.read_csv(file_path, sep='|')
        except:
            print('file not exist in:', folder[i])
    else:
        continue

    # --- KEY LOGIC FOR INTER-FILE DEDUPLICATION ---
    
    # Concatenate the current file's data with all previously processed data
    # (Do NOT use ignore_index=True here, we need separate indices for now)
    
    if final_unique_dfs:
        # Create a temporary DataFrame of ALL data processed so far
        all_prior_data = pd.concat(final_unique_dfs)
        
        # Check the current_df against all prior data. 
        # We only look at the columns that contain the actual data (excluding the index).
        data_columns = current_df.columns
        
        # Identify rows in the current file that are NOT duplicates of PRIOR files
        # The 'indicator=True' is used to identify the source of the merge
        merged = pd.merge(
            current_df, 
            all_prior_data, 
            on=list(data_columns), # Merge on all data columns
            how='left', 
            indicator=True
        )
        
        # Rows in the current file that are *only* in the 'left' (current_df) are unique
        unique_to_current_file = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
        
        # Append the unique data (which includes any intra-file duplicates)
        final_unique_dfs.append(unique_to_current_file)
    else:
        # The first file is added as is (it has no prior files to check against)
        final_unique_dfs.append(current_df)


# 3. Final Concatenation
if final_unique_dfs:
    # Concatenate the final list of DataFrames to produce the result
    final_df = pd.concat(final_unique_dfs, ignore_index=True)

    print("✅ Successfully merged and removed ONLY inter-file duplicates.")
    print("\nFinal DataFrame head (includes intra-file duplicates):")
else:
    print("❌ No valid files were read. The final DataFrame is empty.")

✅ Successfully merged and removed ONLY inter-file duplicates.

Final DataFrame head (includes intra-file duplicates):


In [10]:
final_df

,PATIENT_STUDY_ID,ACCESSION_NUMBER,INTERNAL_EXAM_ID,PATIENT_ID,RISK_FACTOR_CD,RISK_FACTOR_NAME,RISK_SEQUENCE,RELATIONSHIP_CD,PROBLEM_CD,PERSON_AGE,COMMENTS,EXAM_COMPLETED_DATE
0,4334469937,69423837,83749463,1152783,2,Intermediate family history of breast cancer (...,15,Mother,Breast CA,61,NaN,06/25/2020
1,4334469937,69423837,83749463,1152783,C,Personal breast cancer history,3,DUMMY,NaN,0,NaN,06/25/2020
2,4334469937,69511096,83836589,1152783,C,Personal breast cancer history,3,DUMMY,NaN,0,NaN,06/09/2020
3,4334469937,69511096,83836589,1152783,2,Intermediate family history of breast cancer (...,15,Mother,Breast CA,61,NaN,06/09/2020
4,4334469937,69525464,83841700,1152783,C,Personal breast cancer history,3,DUMMY,NaN,0,NaN,06/09/2020
...,...,...,...,...,...,...,...,...,...,...,...,...
379796,4333119414,450097209,73368129,2186707,1,"Weak family history of breast cancer (aunt, gr...",14,Maternal Grandmother,Breast CA,60,NaN,12/27/2022
379797,4334331716,450009554,73370350,1210201,H,History of high risk lesion on previous biopsy...,11,DUMMY,NaN,0,NaN,12/21/2022
379798,4334331716,450009554,73370350,1210201,Q,Post-menopausal patient,2,DUMMY,NaN,0,NaN,12/21/2022
379799,4334331716,450009554,73370350,1210201,Z,Late child bearing (after 30),6,DUMMY,NaN,0,NaN,12/21/2022


In [11]:
final_df['EXAM_COMPLETED_DATE'] = pd.to_datetime(final_df['EXAM_COMPLETED_DATE'], format='%m/%d/%Y')
final_df['EXAM_COMPLETED_DATE'] = final_df['EXAM_COMPLETED_DATE'].dt.strftime('%Y-%m-%d')

In [12]:
final_df.sort_values(by=['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE'], ascending=[True, True], inplace=True, ignore_index=True)

#### <span style="color: blue;">Remove duplicated entries, add a column marked how many duplicates </span> 

**Before drop duplicate = <span style="color: blue;">final_df</span> , after drop duplicate = <span style="color: blue;">unique_df</span>**

In [13]:
unique_df = final_df[extract_cols].copy()

In [14]:
unique_df['duplicate_count'] = unique_df.groupby(unique_df.columns.tolist(), dropna=False).transform('size')

In [15]:
unique_df = unique_df.drop_duplicates(subset=unique_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [16]:
unique_extract_cols = extract_cols + ['duplicate_count']

In [17]:
unique_df[unique_df["PATIENT_STUDY_ID"]==4330018595]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,RISK_FACTOR_CD,RISK_FACTOR_NAME,RISK_SEQUENCE,EXAM_COMPLETED_DATE,duplicate_count
0,4330018595,63027507,0,No family history of breast cancer,13,2019-07-26,1
1,4330018595,63027507,C,Personal breast cancer history,3,2019-07-26,1
2,4330018595,63737104,0,No family history of breast cancer,13,2019-08-19,1
3,4330018595,63737104,C,Personal breast cancer history,3,2019-08-19,1
4,4330018595,60103700,0,No family history of breast cancer,13,2020-06-02,1
5,4330018595,60103700,C,Personal breast cancer history,3,2020-06-02,1
6,4330018595,60695725,0,No family history of breast cancer,13,2020-06-02,1
7,4330018595,60695725,C,Personal breast cancer history,3,2020-06-02,1
8,4330018595,60690108,C,Personal breast cancer history,3,2020-06-02,1
9,4330018595,60690108,0,No family history of breast cancer,13,2020-06-02,1


In [18]:
unique_df['duplicate_count'].unique()

array([1, 3, 4, 2, 5, 6, 7])

In [19]:
unique_df[extract_cols]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,RISK_FACTOR_CD,RISK_FACTOR_NAME,RISK_SEQUENCE,EXAM_COMPLETED_DATE
0,4330018595,63027507,0,No family history of breast cancer,13,2019-07-26
1,4330018595,63027507,C,Personal breast cancer history,3,2019-07-26
2,4330018595,63737104,0,No family history of breast cancer,13,2019-08-19
3,4330018595,63737104,C,Personal breast cancer history,3,2019-08-19
4,4330018595,60103700,0,No family history of breast cancer,13,2020-06-02
...,...,...,...,...,...,...
337136,4339959661,454612462,1,"Weak family history of breast cancer (aunt, gr...",14,2021-11-15
337137,4339959661,453550821,1,"Weak family history of breast cancer (aunt, gr...",14,2021-11-22
337138,4339959661,453550821,U,Nulliparous,4,2021-11-22
337139,4339959661,453550820,1,"Weak family history of breast cancer (aunt, gr...",14,2021-11-22


#### <span style="color: blue;">One hot encoding for RISK_FACTOR_NAME </span> 

**Before = <span style="color: blue;">df = unique_df[extract_cols]</span> , after  = <span style="color: blue;">df_encoded</span>**

In [20]:
df = unique_df[extract_cols]

In [21]:
cols = ['No family history of breast cancer',
       'Personal breast cancer history',
       'Weak family history of breast cancer (aunt, grandmother, cousin)',
       'Late child bearing (after 30)',
       'Very strong family history of breast cancer (mother, sister, daughter, pre-menopause or multiple  post-menopausal first degree relatives)',
       'Family history of ovarian cancer in mother, sister, or daughter',
       'Post-menopausal patient', 'BRCA1 gene mutation', 'Nulliparous',
       'Family history of ovarian cancer in distant blood relatives',
       'Intermediate family history of breast cancer (mother, sister, daughter, post-menopause)',
       'History of high risk lesion on previous biopsy, LCIS, atypical hyperplasia',
       'Previous chest radiation therapy',
       'Other family history of breast cancer', 'BRCA2 gene mutation',
       'Family history unknown for breast cancer',
       'History of ovarian cancer', 'History of endometrial cancer',
       'Not available']

In [22]:
len(cols)

19

In [23]:
grouping_keys = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE']

In [24]:
df_dummies = pd.get_dummies(
    df,
    columns=['RISK_FACTOR_NAME'],
    prefix='',
    prefix_sep=''
)

In [25]:
df_encoded = df_dummies.groupby(grouping_keys, as_index=False)[cols].sum()

In [26]:
final_columns = grouping_keys + cols

In [27]:
df_encoded = df_encoded[final_columns]

In [28]:
df_encoded

,PATIENT_STUDY_ID,ACCESSION_NUMBER,EXAM_COMPLETED_DATE,No family history of breast cancer,Personal breast cancer history,"Weak family history of breast cancer (aunt, grandmother, cousin)",Late child bearing (after 30),"Very strong family history of breast cancer (mother, sister, daughter, pre-menopause or multiple post-menopausal first degree relatives)","Family history of ovarian cancer in mother, sister, or daughter",Post-menopausal patient,BRCA1 gene mutation,Nulliparous,Family history of ovarian cancer in distant blood relatives,"Intermediate family history of breast cancer (mother, sister, daughter, post-menopause)","History of high risk lesion on previous biopsy, LCIS, atypical hyperplasia",Previous chest radiation therapy,Other family history of breast cancer,BRCA2 gene mutation,Family history unknown for breast cancer,History of ovarian cancer,History of endometrial cancer,Not available
0,4330018595,60103700,2020-06-02,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,4330018595,60667802,2020-06-05,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,4330018595,60690108,2020-06-02,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,4330018595,60693047,2020-06-05,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,4330018595,60695725,2020-06-02,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119856,4339959661,78525169,2018-04-16,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
119857,4339959661,453550820,2021-11-22,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
119858,4339959661,453550821,2021-11-22,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
119859,4339959661,454148687,2021-11-10,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0


In [29]:
df_encoded.sort_values(by=['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE'], ascending=[True, True], inplace=True, ignore_index=True)

In [30]:
df_encoded.columns = df_encoded.columns[0:3].tolist() + ['RISK_FACTOR_'+r for r in df_encoded.columns[3:].tolist()]

#### <span style="color: orange;">SAVE </span> 

In [31]:
# output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
# final_df[extract_cols].to_excel(output_file, index=False)

output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
df_encoded.to_excel(output_file, index=False)

### Control

In [32]:
file_names = [
    str(file) + "_controls.txt", 
    str(file) + "_controls.csv",
]
file_names

['risk_factors_controls.txt', 'risk_factors_controls.csv']

In [33]:
study = "Control"
folder = ["R3_3787_Lee_Control_Extract_Files", "R3_3787_Lee_Data_Controls_20240508"]

In [34]:
final_unique_dfs = [] # List to hold the de-duplicated data from each file

# 2. Process each file
for i, file_name in enumerate(file_names):
    file_extension = os.path.splitext(file_name)[1].lower()
    file_path = os.path.join(shared_path, study, folder[i], file_name)
    
    if file_extension == '.csv':
        try:
            current_df = pd.read_csv(file_path, encoding='latin-1')
        except:
            print('file not exist in:', folder[i])
    elif file_extension == '.txt':
        try:
            current_df = pd.read_csv(file_path, sep='|')
        except:
            print('file not exist in:', folder[i])
    else:
        continue

    # --- KEY LOGIC FOR INTER-FILE DEDUPLICATION ---
    
    # Concatenate the current file's data with all previously processed data
    # (Do NOT use ignore_index=True here, we need separate indices for now)
    
    if final_unique_dfs:
        # Create a temporary DataFrame of ALL data processed so far
        all_prior_data = pd.concat(final_unique_dfs)
        
        # Check the current_df against all prior data. 
        # We only look at the columns that contain the actual data (excluding the index).
        data_columns = current_df.columns
        
        # Identify rows in the current file that are NOT duplicates of PRIOR files
        # The 'indicator=True' is used to identify the source of the merge
        merged = pd.merge(
            current_df, 
            all_prior_data, 
            on=list(data_columns), # Merge on all data columns
            how='left', 
            indicator=True
        )
        
        # Rows in the current file that are *only* in the 'left' (current_df) are unique
        unique_to_current_file = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
        
        # Append the unique data (which includes any intra-file duplicates)
        final_unique_dfs.append(unique_to_current_file)
    else:
        # The first file is added as is (it has no prior files to check against)
        final_unique_dfs.append(current_df)


# 3. Final Concatenation
if final_unique_dfs:
    # Concatenate the final list of DataFrames to produce the result
    final_df = pd.concat(final_unique_dfs, ignore_index=True)

    print("✅ Successfully merged and removed ONLY inter-file duplicates.")
    print("\nFinal DataFrame head (includes intra-file duplicates):")
else:
    print("❌ No valid files were read. The final DataFrame is empty.")

✅ Successfully merged and removed ONLY inter-file duplicates.

Final DataFrame head (includes intra-file duplicates):


In [35]:
final_df

,PATIENT_STUDY_ID,ACCESSION_NUMBER,INTERNAL_EXAM_ID,PATIENT_ID,RISK_FACTOR_CD,RISK_FACTOR_NAME,RISK_SEQUENCE,RELATIONSHIP_CD,PROBLEM_CD,PERSON_AGE,COMMENTS,EXAM_COMPLETED_DATE
0,4334585594,68008794,81307232,45497467,3,Very strong family history of breast cancer (m...,16,Maternal Cousin,Breast CA,67,NaN,12/08/2020
1,4334585594,68008794,81307232,45497467,Q,Post-menopausal patient,2,DUMMY,NaN,0,NaN,12/08/2020
2,4334585594,68008794,81307232,45497467,3,Very strong family history of breast cancer (m...,16,Maternal Cousin,Breast CA,65,NaN,12/08/2020
3,4334585594,68008794,81307232,45497467,3,Very strong family history of breast cancer (m...,16,Sister,Breast CA,42,NaN,12/08/2020
4,4334585594,68008794,81307232,45497467,3,Very strong family history of breast cancer (m...,16,Sister,Breast CA,59,NaN,12/08/2020
...,...,...,...,...,...,...,...,...,...,...,...,...
714705,4333191387,459132321,72413116,1227770,Q,Post-menopausal patient,2,DUMMY,NaN,0,NaN,12/19/2022
714706,4333191387,459132326,72413194,1227770,0,No family history of breast cancer,13,DUMMY,NaN,0,NaN,12/19/2022
714707,4333191387,459132326,72413194,1227770,Q,Post-menopausal patient,2,DUMMY,NaN,0,NaN,12/19/2022
714708,4333249051,459109754,72480908,3909876,Q,Post-menopausal patient,2,DUMMY,NaN,0,NaN,12/30/2022


In [36]:
final_df['EXAM_COMPLETED_DATE'] = pd.to_datetime(final_df['EXAM_COMPLETED_DATE'], format='%m/%d/%Y')
final_df['EXAM_COMPLETED_DATE'] = final_df['EXAM_COMPLETED_DATE'].dt.strftime('%Y-%m-%d')

In [37]:
final_df.sort_values(by=['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE'], ascending=[True, True], inplace=True, ignore_index=True)

#### <span style="color: blue;">Remove duplicated entries, add a column marked how many duplicates </span> 

**Before drop duplicate = <span style="color: blue;">final_df</span> , after drop duplicate = <span style="color: blue;">unique_df</span>**

In [38]:
unique_df = final_df[extract_cols].copy()

In [39]:
unique_df['duplicate_count'] = unique_df.groupby(unique_df.columns.tolist(), dropna=False).transform('size')

In [40]:
unique_df = unique_df.drop_duplicates(subset=unique_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [41]:
unique_extract_cols = extract_cols + ['duplicate_count']

In [42]:
unique_df['duplicate_count'].unique()

array([1, 5, 2, 3, 4, 6, 7, 8, 9])

In [43]:
unique_df[unique_extract_cols]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,RISK_FACTOR_CD,RISK_FACTOR_NAME,RISK_SEQUENCE,EXAM_COMPLETED_DATE,duplicate_count
0,4330000534,78048053,Z,Late child bearing (after 30),6,2018-03-28,1
1,4330000534,78048053,0,No family history of breast cancer,13,2018-03-28,1
2,4330000534,64787403,0,No family history of breast cancer,13,2019-05-18,1
3,4330000534,64787403,Z,Late child bearing (after 30),6,2019-05-18,1
4,4330000534,63280481,0,No family history of breast cancer,13,2019-06-20,1
...,...,...,...,...,...,...,...
633575,4339988199,67718778,Q,Post-menopausal patient,2,2021-02-25,1
633576,4339988199,67718778,Z,Late child bearing (after 30),6,2021-02-25,1
633577,4339988199,452549960,0,No family history of breast cancer,13,2022-03-01,1
633578,4339988199,452549960,Q,Post-menopausal patient,2,2022-03-01,1


#### <span style="color: blue;">One hot encoding for RISK_FACTOR_NAME </span> 

**Before = <span style="color: blue;">df = unique_df[extract_cols]</span> , after  = <span style="color: blue;">df_encoded</span>**

In [44]:
df = unique_df[extract_cols]

In [45]:
grouping_keys = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE']

In [46]:
df_dummies = pd.get_dummies(
    df,
    columns=['RISK_FACTOR_NAME'],
    prefix='',
    prefix_sep=''
)

In [47]:
df_encoded = df_dummies.groupby(grouping_keys, as_index=False)[cols].sum()

In [48]:
final_columns = grouping_keys + cols

In [49]:
df_encoded = df_encoded[final_columns]

In [50]:
df_encoded

,PATIENT_STUDY_ID,ACCESSION_NUMBER,EXAM_COMPLETED_DATE,No family history of breast cancer,Personal breast cancer history,"Weak family history of breast cancer (aunt, grandmother, cousin)",Late child bearing (after 30),"Very strong family history of breast cancer (mother, sister, daughter, pre-menopause or multiple post-menopausal first degree relatives)","Family history of ovarian cancer in mother, sister, or daughter",Post-menopausal patient,BRCA1 gene mutation,Nulliparous,Family history of ovarian cancer in distant blood relatives,"Intermediate family history of breast cancer (mother, sister, daughter, post-menopause)","History of high risk lesion on previous biopsy, LCIS, atypical hyperplasia",Previous chest radiation therapy,Other family history of breast cancer,BRCA2 gene mutation,Family history unknown for breast cancer,History of ovarian cancer,History of endometrial cancer,Not available
0,4330000534,60228678,2020-06-01,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,4330000534,60687102,2020-06-10,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,4330000534,63280358,2019-06-20,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,4330000534,63280481,2019-06-20,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,4330000534,63433254,2019-06-20,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291983,4339982047,450915296,2022-10-12,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
291984,4339988199,61697072,2020-02-21,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
291985,4339988199,65403287,2018-11-28,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
291986,4339988199,67718778,2021-02-25,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0


In [51]:
df_encoded.sort_values(by=['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE'], ascending=[True, True], inplace=True, ignore_index=True)

In [52]:
df_encoded.columns = df_encoded.columns[0:3].tolist() + ['RISK_FACTOR_'+r for r in df_encoded.columns[3:].tolist()]

#### <span style="color: orange;">SAVE </span> 

In [53]:
# output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
# final_df[extract_cols].to_excel(output_file, index=False)

output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
df_encoded.to_excel(output_file, index=False)